# Student Performance Analysis

Cleaning, descriptive statistics, derived features and outlier detection for the exam-scores dataset (`StudentsPerformance.csv`).

| Step | What it does |
|---|---|
| 1 | Load the data and inspect its quality |
| 2 | Clean the categorical features (gender, race/ethnicity, parental level of education, lunch, test prep course) |
| 3 | Validate the subject scores |
| 4 | Compute mean, median, mode, standard deviation and quartiles for each subject |
| 5 | Create `total_marks` and `percentage` features |
| 6 | Detect extreme performance outliers across subjects |
| 7 | Export the results |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# ---- settings you may want to change -------------------------------------
DATA_FILE = "StudentsPerformance.csv"   # name of the input CSV
OUTPUT_DIR = Path("output")             # results (CSV files, charts) are saved here
MAX_SCORE = 100                         # maximum marks for one subject
IQR_MULTIPLIER = 1.5                    # Tukey fence width for outlier detection
Z_THRESHOLD = 3.0                       # |z-score| needed to call an outlier "extreme"
# --------------------------------------------------------------------------

OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load and inspect the data

In [ ]:
def locate_data(filename):
    """Find the CSV in the working folder, ./data, or Google Drive (when running in Colab)."""
    try:
        from google.colab import drive          # only exists inside Google Colab
        drive.mount("/content/drive")
    except ImportError:
        pass                                    # running locally - nothing to mount

    candidates = [
        Path(filename),
        Path("data") / filename,
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    searched = "\n  ".join(str(p) for p in candidates)
    raise FileNotFoundError(f"'{filename}' not found. Looked in:\n  {searched}")


raw = pd.read_csv(locate_data(DATA_FILE))
print(f"Loaded {raw.shape[0]:,} rows x {raw.shape[1]} columns\n")
display(raw.head())
raw.info()

In [ ]:
print("Missing values per column:")
print(raw.isnull().sum().to_string())
print("\nDuplicate rows:", raw.duplicated().sum())

## 2. Clean the categorical features

Column names are converted to `snake_case`, then every categorical column is
cleaned in three ways:

1. **Text normalisation** - trim spaces, collapse repeated spaces, ignore case and apostrophe style.
2. **Canonical labels** - each value is mapped to one agreed spelling (e.g. `"Bachelors degree"` and `"bachelor's degree"` both become `"Bachelor's Degree"`). Anything unrecognised is reported and set to missing.
3. **Proper dtype** - columns become `category`; parental education is an *ordered* category from lowest to highest level.

In [ ]:
df = raw.copy()

# tidy column names: "math score" -> "math_score", etc.
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
)
df = df.rename(columns={
    "parental_level_of_education": "parental_education",
    "test_preparation_course": "test_prep_course",
})

SCORE_COLS = ["math_score", "reading_score", "writing_score"]
CATEGORICAL_COLS = ["gender", "race_ethnicity", "parental_education", "lunch", "test_prep_course"]

missing_cols = set(SCORE_COLS + CATEGORICAL_COLS) - set(df.columns)
assert not missing_cols, f"Expected columns not found in the CSV: {missing_cols}"

In [ ]:
# Lookup keys are lower-case with apostrophes removed; values are the clean labels.
CANONICAL_LABELS = {
    "gender": {"female": "Female", "f": "Female", "male": "Male", "m": "Male"},
    "race_ethnicity": {f"group {g.lower()}": f"Group {g}" for g in "ABCDE"},
    "parental_education": {
        "some high school": "Some High School",
        "high school": "High School",
        "some college": "Some College",
        "associates degree": "Associate's Degree",
        "bachelors degree": "Bachelor's Degree",
        "masters degree": "Master's Degree",
    },
    "lunch": {"standard": "Standard", "free/reduced": "Free/Reduced"},
    "test_prep_course": {"none": "None", "completed": "Completed"},
}
EDUCATION_ORDER = ["Some High School", "High School", "Some College",
                   "Associate's Degree", "Bachelor's Degree", "Master's Degree"]


def normalise_key(series):
    """Trim, collapse whitespace, lower-case and drop apostrophes so spelling variants match."""
    return (
        series.astype("string").str.strip()
        .str.replace(r"\s+", " ", regex=True).str.lower()
        .str.replace(r"[\u2019\u2018`\u00b4']", "", regex=True)
    )


before = {c: sorted(df[c].dropna().astype(str).unique()) for c in CATEGORICAL_COLS}

for col in CATEGORICAL_COLS:
    mapped = normalise_key(df[col]).map(CANONICAL_LABELS[col])
    unmapped = sorted(set(df.loc[mapped.isna() & df[col].notna(), col].astype(str)))
    if unmapped:
        print(f"WARNING '{col}': unrecognised labels set to missing -> {unmapped}")
    df[col] = mapped.astype("category")

df["parental_education"] = df["parental_education"].cat.set_categories(EDUCATION_ORDER, ordered=True)

for col in CATEGORICAL_COLS:
    print(f"{col}\n  before: {before[col]}\n  after : {list(df[col].cat.categories)}\n")

In [ ]:
for col in CATEGORICAL_COLS:
    print(df[col].value_counts(dropna=False).to_string(), "\n")

## 3. Validate scores and remove unusable rows

Scores must be numeric and lie between 0 and `MAX_SCORE`. Invalid scores become
missing; rows with any missing value (and exact duplicates) are then dropped. The
dataset's index still refers to the row number in the original CSV.

In [ ]:
for col in SCORE_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid = (df[SCORE_COLS] >= 0) & (df[SCORE_COLS] <= MAX_SCORE)   # NaN compares False -> counted as invalid
print(f"Invalid or non-numeric scores (outside 0-{MAX_SCORE}):")
print((~valid).sum().to_string())
df[SCORE_COLS] = df[SCORE_COLS].where(valid)

n_start = len(df)
df = df.dropna(subset=CATEGORICAL_COLS + SCORE_COLS)
n_missing = n_start - len(df)

for col in SCORE_COLS:                       # restore integer dtype when every score is a whole number
    if (df[col] % 1 == 0).all():
        df[col] = df[col].astype(int)

n_before_dupes = len(df)
df = df.drop_duplicates()
n_dupes = n_before_dupes - len(df)

print(f"\nRows dropped for missing/invalid values: {n_missing}")
print(f"Duplicate rows dropped                 : {n_dupes}")
print(f"Clean dataset                          : {len(df):,} rows")

## 4. Descriptive statistics for the subject scores

`std` is the sample standard deviation (n - 1). `median (Q2)` is the 50th percentile;
`Q1`/`Q3` are the 25th/75th percentiles and `IQR = Q3 - Q1`. If a subject has several
modes, the smallest is shown.

In [ ]:
def score_summary(data, cols):
    x = data[cols]
    q1, q2, q3 = x.quantile(0.25), x.quantile(0.50), x.quantile(0.75)
    return pd.DataFrame({
        "count": x.count(),
        "mean": x.mean(),
        "median (Q2)": q2,
        "mode": x.mode().iloc[0],
        "std": x.std(),
        "min": x.min(),
        "Q1 (25%)": q1,
        "Q3 (75%)": q3,
        "max": x.max(),
        "IQR": q3 - q1,
        "skewness": x.skew(),
    }).round(2)


score_stats = score_summary(df, SCORE_COLS)
display(score_stats.T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colours = sns.color_palette("Set2", 3)

for ax, col, colour in zip(axes, SCORE_COLS, colours):
    sns.histplot(df[col], bins=20, kde=True, color=colour, ax=ax)
    ax.axvline(df[col].mean(), color="black", linestyle="--", label=f"mean {df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="crimson", linestyle=":", label=f"median {df[col].median():.1f}")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Score")
    ax.legend()

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "score_distributions.png", dpi=150)
plt.show()

## 5. Calculated features: total marks and percentage

* `total_marks` = math + reading + writing
* `percentage` = `total_marks` / (`MAX_SCORE` x number of subjects) x 100
* `performance_band` groups the percentage into bands for easy comparison

In [ ]:
TOTAL_MAX = MAX_SCORE * len(SCORE_COLS)          # 300 for three subjects
BAND_BINS = [0, 40, 50, 60, 75, 90, 100.0001]     # lower edges are inclusive
BAND_LABELS = ["Below 40", "40-49", "50-59", "60-74", "75-89", "90-100"]

df["total_marks"] = df[SCORE_COLS].sum(axis=1)
df["percentage"] = (df["total_marks"] / TOTAL_MAX * 100).round(2)
df["performance_band"] = pd.cut(df["percentage"], bins=BAND_BINS, labels=BAND_LABELS, right=False)

display(df[SCORE_COLS + ["total_marks", "percentage", "performance_band"]].head())
display(df[["total_marks", "percentage"]].describe().round(2).T)
print(df["performance_band"].value_counts().sort_index().to_string())

## 6. Detect extreme performance outliers

Each subject (and `total_marks`) is checked with two complementary rules:

* **IQR / Tukey fences** - a score is an *outlier* if it is below `Q1 - 1.5 x IQR` or above `Q3 + 1.5 x IQR`.
* **Z-score** - a score is an *extreme outlier* if it is an IQR outlier **and** lies at least 3 standard deviations from the mean.

Outliers are **flagged, not removed**, so you can decide what to do with them.

In [ ]:
OUTLIER_COLS = SCORE_COLS + ["total_marks"]


def detect_outliers(series):
    """Return low/high/extreme boolean masks and the lower/upper IQR fences."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr
    z = (series - series.mean()) / series.std()
    low, high = series < lower, series > upper
    extreme = (low | high) & (z.abs() >= Z_THRESHOLD)
    return low, high, extreme, lower, upper


flags, rows = {}, []
for col in OUTLIER_COLS:
    low, high, extreme, lower, upper = detect_outliers(df[col])
    flags[col] = {"low": low, "high": high, "extreme": extreme}
    rows.append({
        "measure": col,
        "lower_fence": round(lower, 2),
        "upper_fence": round(upper, 2),
        "lowest": df[col].min(),
        "highest": df[col].max(),
        "low_outliers": int(low.sum()),
        "high_outliers": int(high.sum()),
        "extreme_outliers": int(extreme.sum()),
    })

outlier_summary = pd.DataFrame(rows).set_index("measure")
display(outlier_summary)

In [ ]:
def label_subjects(mask):
    """For each student, list the subjects flagged in a boolean mask DataFrame."""
    return mask.apply(lambda r: ", ".join(c.replace("_score", "") for c in r.index[r.values]), axis=1)


low_mask = pd.DataFrame({c: flags[c]["low"] for c in SCORE_COLS})
high_mask = pd.DataFrame({c: flags[c]["high"] for c in SCORE_COLS})
extreme_mask = pd.DataFrame({c: flags[c]["extreme"] for c in SCORE_COLS})
any_mask = low_mask | high_mask

df["n_outlier_subjects"] = any_mask.sum(axis=1)
df["outlier_subjects"] = label_subjects(any_mask)
df["extreme_subjects"] = label_subjects(extreme_mask)
df["outlier_direction"] = np.select(
    [low_mask.any(axis=1) & high_mask.any(axis=1), low_mask.any(axis=1), high_mask.any(axis=1)],
    ["mixed", "low", "high"],
    default="none",
)
df["total_marks_outlier"] = flags["total_marks"]["low"] | flags["total_marks"]["high"]

print("Students by number of outlier subjects:")
print(df["n_outlier_subjects"].value_counts().sort_index().to_string())

In [ ]:
report_cols = (CATEGORICAL_COLS + SCORE_COLS +
               ["total_marks", "percentage", "n_outlier_subjects",
                "outlier_subjects", "outlier_direction", "extreme_subjects"])

flagged = (df["n_outlier_subjects"] > 0) | df["total_marks_outlier"]
outlier_report = (
    df.loc[flagged, report_cols]
    .sort_values(["n_outlier_subjects", "percentage"], ascending=[False, True])
)
print(f"{len(outlier_report)} students flagged ({len(outlier_report) / len(df):.1%} of {len(df):,})\n")

extreme_students = outlier_report[outlier_report["extreme_subjects"] != ""]
print(f"Extreme outliers (IQR rule + |z| >= {Z_THRESHOLD}): {len(extreme_students)} students")
display(extreme_students)

print("\nAll flagged students (top 15 by severity):")
display(outlier_report.head(15))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={"height_ratios": [3, 1.2]})

sns.boxplot(data=df[SCORE_COLS], orient="h", whis=IQR_MULTIPLIER, palette="Set2", ax=axes[0])
axes[0].set_title("Subject scores - points beyond the whiskers are outliers")
axes[0].set_xlabel("Score")

sns.boxplot(x=df["total_marks"], orient="h", whis=IQR_MULTIPLIER, color="lightsteelblue", ax=axes[1])
axes[1].set_title("Total marks")
axes[1].set_xlabel(f"Total marks (out of {TOTAL_MAX})")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "outlier_boxplots.png", dpi=150)
plt.show()

## 7. Export results and summary

In [ ]:
df.to_csv(OUTPUT_DIR / "students_cleaned.csv", index_label="row_id")
score_stats.to_csv(OUTPUT_DIR / "score_statistics.csv")
outlier_summary.to_csv(OUTPUT_DIR / "outlier_summary.csv")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index_label="row_id")

print("Saved to", OUTPUT_DIR.resolve())
for f in sorted(OUTPUT_DIR.iterdir()):
    print("  -", f.name)

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Students analysed : {len(df):,}")
print(f"Average percentage: {df['percentage'].mean():.2f}%  (median {df['percentage'].median():.2f}%)")
print(f"Range             : {df['percentage'].min():.2f}% to {df['percentage'].max():.2f}%\n")

for col in SCORE_COLS:
    s = score_stats.loc[col]
    n_low, n_high, n_ext = (int(outlier_summary.loc[col, k]) for k in ("low_outliers", "high_outliers", "extreme_outliers"))
    print(f"{col.replace('_', ' ').title():<14} mean {s['mean']:.1f} | median {s['median (Q2)']:.1f} | "
          f"std {s['std']:.1f} | outliers {n_low + n_high} (low {n_low}, high {n_high}) | extreme {n_ext}")

multi = int((df["n_outlier_subjects"] >= 2).sum())
print(f"\nStudents that are outliers in 2+ subjects: {multi}")